<a href="https://colab.research.google.com/github/Berkaycetinn/log-analyzer/blob/main/veri_kesfi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone --depth 1 https://github.com/logpai/loghub.git
!ls loghub/Hadoop
!head -n 20 loghub/Hadoop/Hadoop_2k.log

Cloning into 'loghub'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 91 (delta 8), reused 36 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.09 MiB | 5.72 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Hadoop_2k.log		      Hadoop_2k.log_templates.csv
Hadoop_2k.log_structured.csv  README.md
2015-10-18 18:01:47,978 INFO [main] org.apache.hadoop.mapreduce.v2.app.MRAppMaster: Created MRAppMaster for application appattempt_1445144423722_0020_000001
2015-10-18 18:01:48,963 INFO [main] org.apache.hadoop.mapreduce.v2.app.MRAppMaster: Executing with tokens:
2015-10-18 18:01:48,963 INFO [main] org.apache.hadoop.mapreduce.v2.app.MRAppMaster: Kind: YARN_AM_RM_TOKEN, Service: , Ident: (appAttemptId { application_id { id: 20 cluster_timestamp: 1445144423722 } attemptId: 1 } keyId: -127633188)
2015-10-18 18:01:49,228 INFO [main] org.apache.hadoop.mapreduce.v2.app.MRAppMaster:

In [2]:
import re
import pandas as pd

pattern = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2},\d{3}) "
    r"(?P<level>[A-Z]+) \[(?P<thread>[^\]]*)\] (?P<component>\S+): (?P<message>.*)$"
)

rows, unmatched = [], []
with open("loghub/Hadoop/Hadoop_2k.log", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        m = pattern.match(line)
        if m:
            rows.append(m.groupdict())
        else:
            unmatched.append(line)

df = pd.DataFrame(rows)
print("Eşleşen satır:", len(df))
print("Eşleşmeyen satır:", len(unmatched))
print(unmatched[:5])

Eşleşen satır: 2000
Eşleşmeyen satır: 0
[]


In [3]:
print(df["level"].value_counts())
print("\nBenzersiz mesaj sayısı:", df["message"].nunique())
print("\nEn çok log yazan 10 sınıf:")
print(df["component"].value_counts().head(10))

level
INFO     1040
WARN      808
ERROR     150
FATAL       2
Name: count, dtype: int64

Benzersiz mesaj sayısı: 734

En çok log yazan 10 sınıf:
component
org.apache.hadoop.ipc.Client                                               622
org.apache.hadoop.mapreduce.v2.app.rm.RMContainerAllocator                 457
org.apache.hadoop.hdfs.LeaseRenewer                                        326
org.apache.hadoop.mapred.TaskAttemptListenerImpl                           314
org.apache.hadoop.mapreduce.v2.app.job.impl.TaskAttemptImpl                 60
org.apache.hadoop.yarn.util.RackResolver                                    39
org.apache.hadoop.mapreduce.v2.app.launcher.ContainerLauncherImpl           37
org.apache.hadoop.mapreduce.v2.app.job.impl.TaskImpl                        24
org.apache.hadoop.mapreduce.v2.app.rm.RMContainerRequestor                  17
org.apache.hadoop.yarn.client.api.impl.ContainerManagementProtocolProxy     14
Name: count, dtype: int64


In [4]:
errors = df[df["level"].isin(["ERROR", "WARN", "FATAL"])]
print("ERROR/WARN/FATAL sayısı:", len(errors))
pd.set_option("display.max_colwidth", 200)
errors[["time", "level", "component", "message"]].head(15)

ERROR/WARN/FATAL sayısı: 960


,time,level,component,message
667,"18:04:11,034",ERROR,org.apache.hadoop.mapreduce.v2.app.rm.RMContainerAllocator,Container complete event for unknown container id container_1445144423722_0020_01_000012
847,"18:05:27,570",WARN,org.apache.hadoop.ipc.Client,Address change detected. Old: msra-sa-41/10.190.173.170:9000 New: msra-sa-41:9000
848,"18:05:27,570",WARN,org.apache.hadoop.hdfs.LeaseRenewer,Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 30 seconds. Will retry shortly ...
849,"18:05:28,570",WARN,org.apache.hadoop.ipc.Client,Address change detected. Old: msra-sa-41/10.190.173.170:9000 New: msra-sa-41:9000
850,"18:05:28,570",WARN,org.apache.hadoop.hdfs.LeaseRenewer,Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 31 seconds. Will retry shortly ...
851,"18:05:29,570",WARN,org.apache.hadoop.ipc.Client,Address change detected. Old: msra-sa-41/10.190.173.170:9000 New: msra-sa-41:9000
852,"18:05:29,570",WARN,org.apache.hadoop.hdfs.LeaseRenewer,Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 32 seconds. Will retry shortly ...
853,"18:05:30,570",WARN,org.apache.hadoop.ipc.Client,Address change detected. Old: msra-sa-41/10.190.173.170:9000 New: msra-sa-41:9000
854,"18:05:30,570",WARN,org.apache.hadoop.hdfs.LeaseRenewer,Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 33 seconds. Will retry shortly ...
855,"18:05:31,570",WARN,org.apache.hadoop.ipc.Client,Address change detected. Old: msra-sa-41/10.190.173.170:9000 New: msra-sa-41:9000


In [5]:
templates = pd.read_csv("loghub/Hadoop/Hadoop_2k.log_templates.csv")
print("Gerçek template sayısı:", len(templates))
templates.head(20)

Gerçek template sayısı: 114


,EventId,EventTemplate
0,E1,<*> failures on node MININT-<*>
1,E2,Added attempt_<*> to list of failed maps
2,E3,Added filter AM_PROXY_FILTER (class=org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter) to context mapreduce
3,E4,Added filter AM_PROXY_FILTER (class=org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter) to context static
4,E5,Added global filter 'safety' (class=org.apache.hadoop.http.HttpServer2$QuotingInputFilter)
5,E6,Adding #<*> tokens and #<*> secret keys for NM use for launching container
6,E7,Adding job token for job_<*> to jobTokenSecretManager
7,E8,adding path spec: /<*>/*
8,E9,Adding protocol org.apache.hadoop.mapreduce.v2.api.MRClientProtocolPB to the server
9,E10,Address change detected. Old: <*>/<*>:<*> New: <*>:<*>


In [6]:
def mask(msg):
    msg = re.sub(r"\b(application|appattempt|container|job|task|attempt)_\w+", r"\1_<ID>", msg)
    msg = re.sub(r"\d+\.\d+\.\d+\.\d+", "<IP>", msg)
    msg = re.sub(r"\b\d+\b", "<NUM>", msg)
    return msg

df["quick_template"] = df["message"].apply(mask)

print("Benzersiz mesaj:", df["message"].nunique())
print("Kaba template:", df["quick_template"].nunique())

errs = df[df["level"].isin(["ERROR", "WARN", "FATAL"])]
print("\nERROR/WARN içinde en sık 10 template:")
print(errs["quick_template"].value_counts().head(10).to_string())

Benzersiz mesaj: 734
Kaba template: 148

ERROR/WARN içinde en sık 10 template:
quick_template
Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa-<NUM>:<NUM>                                                                                                                                                                                                                                       476
Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for <NUM> seconds.  Will retry shortly ...                                                                                                                                                                                                            326
ERROR IN CONTACTING RM.                                                                                                                                                                                                                                                               

In [7]:
key = df["quick_template"]
top = errs["quick_template"].value_counts().head(3).index
rare = errs["quick_template"].value_counts()
rare = rare[rare <= 2].index

def timeline(templates):
    sub = df[key.isin(templates)]
    return (sub.groupby("quick_template")
               .agg(adet=("message", "size"), ilk=("time", "min"), son=("time", "max"))
               .sort_values("ilk"))

pd.set_option("display.max_colwidth", 70)
print(timeline(list(top) + [t for t in rare if "NoRoute" in t]))

                                                                        adet  \
quick_template                                                                 
Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa...   476   
Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for...   326   
ERROR IN CONTACTING RM.                                                  147   
Task: attempt_<ID> - exited : java.net.NoRouteToHostException: No R...     2   

                                                                                 ilk  \
quick_template                                                                         
Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa...  18:05:27,570   
Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for...  18:05:27,570   
ERROR IN CONTACTING RM.                                                 18:06:01,840   
Task: attempt_<ID> - exited : java.net.NoRouteToHostException: No R...  18:06:2